# ML09 · CNN 卷積神經網路

用卷積（convolution）與池化（pooling）讓神經網路「看懂」影像，並親手建一個小型卷積神經網路（convolutional neural network, CNN）分類 MNIST 手寫數字。

## 學習目標
- 理解為何影像分類用 CNN 比多層感知器（multilayer perceptron, MLP）更合適。
- 建立卷積與池化的直覺：局部感受野（receptive field）、權重共享（weight sharing）、降採樣（downsampling）。
- 分清楚通道（channel）與特徵圖（feature map）的概念與形狀變化。
- 會用 torchvision 載入 MNIST 手寫數字資料集（本書允許連網下載）。
- 能組裝一個小型 CNN，完成訓練並以準確率（accuracy）評估分類結果。

In [ ]:
# 概念：先做好繪圖與亂數環境設定，後面所有範例共用
import numpy as np
import matplotlib.pyplot as plt

# 固定亂數種子，讓每次執行造出的假資料與結果一致（方便對照講解）
rng = np.random.default_rng(0)

# 讓圖以灰階呈現，符合本書多為單通道影像的情境
plt.rcParams["image.cmap"] = "gray"

print("環境設定完成，numpy 版本:", np.__version__)

## 為何影像不用 MLP：CNN 的動機

影像是有空間結構（spatial structure）的資料：相鄰像素（pixel）彼此高度相關，一條邊、一個轉角都是由「附近一小塊」像素共同構成。

MLP 會先把影像攤平（flatten）成一長串向量，這一步丟掉了「哪些像素互相相鄰」的資訊；而且全連接層會造成參數爆炸（parameter explosion），一張小圖就需要極大量的權重。

CNN 改用「在影像上滑動的小濾鏡」保留空間結構並共享權重，因此更省參數、也更穩定。

In [ ]:
# 概念：影像攤平成向量後，相鄰像素的關係（局部相關性 local correlation）就消失了
import numpy as np

# 造一張 4x4 的小灰階方格圖：左半邊亮(1)、右半邊暗(0)，中間有一條清楚的垂直邊
image = np.array([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
])

print("原圖 shape:", image.shape, " 一眼可見第 1、2 欄之間有一條垂直邊")
print(image)

flat = image.flatten()             # 攤平成一維向量，模仿 MLP 的輸入
print("\n攤平後 shape:", flat.shape)
print("攤平後:", flat)
# 注意：攤平後第 1 欄與第 2 欄原本相鄰的像素，在向量裡被拆到不連續的位置，空間關係不見了

# 估算全連接的參數量：把這張圖接到一個同樣大小的隱藏層需要幾個權重
n_pixels = image.size
fc_params = n_pixels * n_pixels    # 每個輸入接每個輸出，權重數是像素數的平方
print("\n全連接權重數（16->16）:", fc_params, "  影像一大，這個數字會爆炸")

## 卷積的直覺：濾鏡與局部特徵

卷積就是拿一個小濾鏡（kernel / filter）在影像上滑動（sliding window），每滑到一個位置就做一次「局部加權相加」。

關鍵在於整張圖用的是同一組濾鏡權重（權重共享 weight sharing），所以同一個花樣不管出現在哪裡都能被偵測到。一個濾鏡能看到的那一小塊輸入範圍，稱為感受野（receptive field）。

設計成「中間正、兩側負」的濾鏡，就能做邊緣偵測（edge detection）：在數值平坦處輸出接近 0，在明暗交界處輸出明顯偏大。

In [ ]:
# 概念：手刻一個 3x3 邊緣濾鏡，用滑動視窗掃過整張圖（這就是卷積）
import numpy as np

# 造一張 6x6 小圖：左半亮、右半暗，中央一條垂直邊
image = np.zeros((6, 6))
image[:, :3] = 1.0                 # 左半邊設為亮

# 垂直邊緣濾鏡：左欄正、右欄負，遇到「左亮右暗」會得到大正值
kernel = np.array([
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0],
])

k = kernel.shape[0]                # 濾鏡邊長（3）
out_h = image.shape[0] - k + 1     # 不補邊時，輸出每軸縮短 (k-1)
out_w = image.shape[1] - k + 1
feature_map = np.zeros((out_h, out_w))

for i in range(out_h):
    for j in range(out_w):
        patch = image[i:i + k, j:j + k]        # 取出濾鏡當前覆蓋的局部視窗
        feature_map[i, j] = np.sum(patch * kernel)  # 局部加權相加，得到這個位置的響應

print("原圖:")
print(image)
print("\n特徵圖（邊緣響應）shape:", feature_map.shape)
print(np.round(feature_map, 1))
# 技巧：絕對值大的位置就是被「點亮」的邊；平坦區域接近 0
print("\n被偵測為邊的最大響應:", feature_map.max())

## 通道與特徵圖：形狀怎麼變

通道（channel）是影像在「深度」方向的層數：灰階圖只有 1 個通道，彩色圖有紅綠藍 3 個通道。

一個卷積層通常用多個濾鏡，每個濾鏡掃過輸入後各自輸出一張特徵圖（feature map）。有幾個濾鏡，輸出就有幾個通道。讀懂 CNN 的關鍵，就是追蹤張量形狀 (通道, 高, 寬) 怎麼隨層變化。

形狀：輸入 (C_in, H, W) 經過 C_out 個濾鏡，輸出變成 (C_out, H', W')。

In [ ]:
# 概念：一張單通道圖經過 2 個濾鏡，會變成 2 張特徵圖（通道數 1 -> 2）
import numpy as np

# 造一張 1 通道、5x5 的小圖，形狀寫成 (C, H, W) 以符合 CNN 慣例
image = rng.uniform(0, 1, size=(1, 5, 5))

# 準備 2 個不同的 3x3 濾鏡：一個偵測垂直邊、一個偵測水平邊
vertical = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)
horizontal = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=float)
kernels = [vertical, horizontal]

def conv2d_single(channel, kernel):
    # 對單一通道做一次卷積，回傳一張特徵圖
    k = kernel.shape[0]
    out_h = channel.shape[0] - k + 1
    out_w = channel.shape[1] - k + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = np.sum(channel[i:i + k, j:j + k] * kernel)
    return out

single_channel = image[0]                 # 取出唯一的輸入通道
feature_maps = [conv2d_single(single_channel, k) for k in kernels]
output = np.stack(feature_maps, axis=0)   # 把 2 張特徵圖疊成 (C_out, H', W')

print("輸入形狀 (C_in, H, W):", image.shape)
print("濾鏡數量 (= 輸出通道數):", len(kernels))
print("輸出形狀 (C_out, H', W'):", output.shape)
print("\n形狀對照：通道 1 -> 2，空間 5x5 -> {}x{}".format(output.shape[1], output.shape[2]))

## 池化與降採樣：抓重點、縮尺寸

池化（pooling）在特徵圖的小區塊裡只留一個代表值，藉此降採樣（downsampling）：縮小特徵圖、降低後續計算量。

最大池化（max pooling）取區塊內最大值，保留「最強的響應」；平均池化（average pooling）取平均值。池化還帶來平移不變性（translation invariance）：花樣在小區塊內輕微位移，輸出仍大致相同。

形狀：用 2x2、步幅 2 的池化，高與寬各約縮為一半。

In [ ]:
# 概念：對一張 4x4 特徵圖做 2x2 最大池化，比較池化前後的數值與尺寸
import numpy as np

# 造一張 4x4 的假特徵圖（數值代表各位置的響應強度）
feature_map = np.array([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 6.0, 1.0, 2.0],
    [7.0, 2.0, 9.0, 1.0],
    [0.0, 4.0, 3.0, 8.0],
])

pool = 2                              # 池化視窗邊長，同時也是步幅 stride
out_h = feature_map.shape[0] // pool  # 不重疊地切，尺寸整除縮小
out_w = feature_map.shape[1] // pool
pooled = np.zeros((out_h, out_w))

for i in range(out_h):
    for j in range(out_w):
        block = feature_map[i * pool:(i + 1) * pool, j * pool:(j + 1) * pool]
        pooled[i, j] = block.max()    # 最大池化：只留區塊內最強的響應

print("池化前 shape:", feature_map.shape)
print(feature_map)
print("\n池化後 shape:", pooled.shape, " 高與寬各縮為一半")
print(pooled)

## 載入 MNIST 資料集

MNIST 是經典的手寫數字資料集，每張是 28x28 的灰階圖，標籤（label）是 0 到 9。我們用 torchvision 下載它，這是本書唯一一次允許的連網下載。

影像需經過資料轉換（transform）中的張量化（to-tensor），轉成形狀 (1, 28, 28) 的張量；再交給資料載入器（DataLoader）依批次（batch）分批餵入模型。

為什麼用批次：一次處理一小批樣本，比逐筆更快也更穩定。

In [ ]:
# 概念：用 torchvision 下載 MNIST，並把影像轉成 (1, 28, 28) 張量
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ToTensor 會把 0-255 的像素縮到 0-1，並排成 (通道, 高, 寬) 的張量
transform = transforms.ToTensor()

# 注意：第一次執行會連網下載到 ./data，之後會直接讀本機快取
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# DataLoader 負責分批與打亂；訓練要打亂、測試不需要
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

image, label = train_set[0]            # 取出第一筆樣本檢視
print("訓練樣本數:", len(train_set), " 測試樣本數:", len(test_set))
print("單張影像 shape:", tuple(image.shape), " 標籤:", label)
print("像素值範圍:", float(image.min()), "到", float(image.max()))

## 組一個小 CNN 並訓練

把前面學到的零件接起來：兩層「卷積 + 激活函數（activation, ReLU）+ 池化」負責抽特徵，最後一層全連接層（fully connected layer）做分類。

分類用交叉熵損失（cross-entropy loss），直覺式為 loss = - Σ y · log(ŷ)：模型對正確類別給的機率越低，懲罰越大。訓練迴圈（training loop）每一步都走「前向 -> 算損失 -> 反向傳播（backpropagation）-> 更新權重」。

形狀：(1, 28, 28) 經兩次卷積與池化後縮小，攤平後接全連接層輸出 10 個類別分數。

In [ ]:
# 概念：定義一個兩層卷積 + 一層全連接的小 CNN，並走完整的訓練迴圈
import torch
import torch.nn as nn

torch.manual_seed(0)                   # 固定權重初始化，讓訓練結果可重現

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 第一層卷積：輸入 1 通道，輸出 8 張特徵圖；padding=1 讓空間尺寸不變
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
        # 第二層卷積：通道數 8 -> 16，繼續抽更抽象的特徵
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)   # 2x2 池化，每用一次空間尺寸減半
        self.relu = nn.ReLU()
        # 兩次池化把 28x28 縮成 7x7；16 通道攤平後接到 10 個類別
        self.fc = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.flatten(start_dim=1)                # 保留 batch 維，其餘攤平成向量
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()                 # 內含 softmax，直接吃類別分數
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 2                                       # 為求快速示範只跑少數輪
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                      # 清掉上一步殘留的梯度
        outputs = model(images)                    # 前向：得到 10 類分數
        loss = criterion(outputs, labels)          # 算交叉熵損失
        loss.backward()                            # 反向傳播：算各權重的梯度
        optimizer.step()                           # 依梯度更新權重
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{n_epochs}  平均訓練損失: {avg_loss:.4f}")

## 評估準確率與看錯在哪

訓練損失只說明「在看過的資料上學得如何」，真正的表現要用沒看過的測試集（test set）衡量。準確率（accuracy）就是預測（prediction）對的比例。

只看一個準確率數字不夠：把模型「認錯成什麼」攤開來看，做錯誤分析（error analysis），更能找到改進方向。挑幾張預測錯的圖，對照真實值與預測值，往往一眼就看出模型在哪裡混淆（confusion）。

In [ ]:
# 概念：在測試集上算整體準確率，並挑幾張預測錯的圖印出真實 vs 預測
import torch

model.eval()                                       # 切到評估模式（關掉只在訓練用的行為）
correct, total = 0, 0
wrong_examples = []                                # 收集少數錯例供檢視

with torch.no_grad():                              # 評估不需算梯度，省記憶體也更快
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)              # 取分數最高的類別當預測
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        # 蒐集前幾個錯例：真實與預測不同的樣本
        if len(wrong_examples) < 4:
            mismatch = (preds != labels).nonzero(as_tuple=True)[0]
            for idx in mismatch:
                if len(wrong_examples) >= 4:
                    break
                wrong_examples.append((images[idx].cpu(), labels[idx].item(), preds[idx].item()))

accuracy = correct / total
print(f"測試集準確率: {accuracy:.4f}  （隨機猜只有約 0.10）")

# 把錯例畫出來，標出真實值與預測值
fig, axes = plt.subplots(1, len(wrong_examples), figsize=(2.2 * len(wrong_examples), 2.4))
for ax, (img, true_label, pred_label) in zip(np.atleast_1d(axes), wrong_examples):
    ax.imshow(img.squeeze(0))                      # 去掉通道維才能當 2D 灰階圖畫
    ax.set_title(f"true {true_label} / pred {pred_label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），CNN 題沿用前面下載好的 MNIST。完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）樓層平面圖的邊緣偵測（整合：卷積 + 特徵圖）
#   情境：你有一張自造的小型建築樓層平面格點圖（牆=1、空地=0），想自動找出牆的邊界。
#   要求：
#     1. 用 numpy 造一張 8x8 的平面圖，把牆排成一個矩形外框（外圈為 1、內部為 0）。
#     2. 設計一個 3x3 邊緣濾鏡，對整張圖做卷積，輸出一張特徵圖。
#     3. 印出原圖與特徵圖，並指出哪些位置被偵測為「邊」（響應絕對值偏大）。
#   驗收：應看到特徵圖在牆與空地的交界處數值明顯偏高，牆內部與空地內部接近 0。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）容積分布圖的卷積加池化管線（整合：卷積 + 通道 + 池化 + 形狀）
#   情境：一個城市街區被切成格網，每格記錄自造的容積率（floor area ratio），
#         你想做一條「卷積 -> 池化」的特徵管線並追蹤形狀變化。
#   要求：
#     1. 用 numpy 造一張 1 通道、16x16 的容積率格網（數值高低代表開發強度）。
#     2. 用 2 個不同濾鏡做卷積，得到 2 張特徵圖，印出形狀 (2, 高, 寬)。
#     3. 對這 2 張特徵圖各做 2x2 最大池化，再次印出形狀，並用一句話說明尺寸為何減半。
#   驗收：應看到通道數從 1 變 2、空間尺寸經池化後約縮為一半，形狀變化前後對得起來。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）用小 CNN 分類建物縮圖並做錯誤分析（整合：MNIST 載入 + CNN 訓練 + 準確率 + 錯誤檢視）
#   情境：把 MNIST 手寫數字當作「建物用途代碼」的縮圖代理資料（數字 0-9 代表 10 種用途分區），
#         訓練一個小 CNN 自動辨識代碼。
#   要求：
#     1. 沿用前面下載的 MNIST，建一個含卷積、池化與全連接層的小 CNN，訓練數個 epoch，
#        在測試集上計算整體準確率。
#     2. 統計測試集上「真實類別 -> 被預測成哪一類」，找出最常被認錯的一對代碼。
#        （技巧：可用一個 10x10 的計數表，列為真實、欄為預測，累加非對角線最大的格子。）
#     3. 挑 1-2 張該錯誤的縮圖印出真實值與預測值，並提出一句改進想法（如加一層卷積或增訓練輪數）。
#   驗收：應看到測試準確率明顯高於隨機猜（遠高於 10%），
#         並能指認出一組最易混淆的類別與一張具體錯例。
# TODO: 在這裡完成


## 小結

- 影像有空間結構，MLP 把它攤平會丟掉相鄰像素的關係且參數爆炸；CNN 用滑動的小濾鏡保留結構並共享權重，更省更穩。
- 卷積是拿同一組濾鏡權重在影像上滑動做局部加權，能偵測不管出現在哪的局部花樣（如邊緣）。
- 一個卷積層用多個濾鏡，各產生一張特徵圖；讀懂 CNN 的關鍵是追蹤 (通道, 高, 寬) 的形狀變化。
- 池化在小區塊內只留代表值（最大池化取最大），縮小特徵圖、降低計算量，並帶來對微小位移的不變性。
- torchvision 能載入 MNIST，影像被轉成 (1, 28, 28) 張量並用 DataLoader 分批餵入模型。
- 一個小 CNN 由「卷積 + ReLU + 池化」堆疊加全連接層組成，訓練走「前向 -> 算交叉熵損失 -> 反向 -> 更新」。
- 用沒看過的測試集算準確率才公平；再做錯誤分析看模型把哪些類別認錯，比單一數字更有洞見。